# Stage 3 corpus

This is the notebook sequence that built the 200-problem pilot. Run it on the GPU pod, with the volume mounted at `/workspace`. Do not run it on the laptop: the leak scan downloads the LiveCodeBench parquet, and the grade cell loads the Stage 2 checkpoint.

`s3/filters.py` is the same three text rules, tested on toy rows. The pod did not import that file. These cells are the ones that ran.

Recorded results, so a later walkthrough can show the numbers without re-grading:

- EvolveCoder: 21,642 rows. Empty tests 0. Duplicate statements 0.
- LiveCodeBench v6: 1,055 problems, 2023-05-07 to 2025-04-06. Prefix copies 0. Shared-phrase copies 946. Kept 20,696.
- Diagnostic slice: 40 ids, seed `Random(1)`, starting `TACO_5353` and ending `APPS_2555`. Train pool 20,656. The nine peeked ids stay in train.
- Easy-first grade: `finished keeps 200 graded 558`. Frozen file had `keeps 200 missing [] eval_leaks []`.
- Private dataset: `syedamaan/rufus-s3-coding-pilot`.

The first split cell read a stale id file, because the leak cell had not closed its write. That split (kept 21,642, eval starting `APPS_507`) was discarded. The split cell below closes the file and is the one that stuck.


## Load EvolveCoder

Printed `21642`, first id `TACO_1`, 67 tests. The kernel must still hold `ds` for every later cell.


In [ ]:
import os
os.environ["HF_HOME"] = "/workspace/hf-cache"
os.environ["HF_HUB_CACHE"] = "/workspace/hf-cache/hub"
from datasets import load_dataset

ds = load_dataset("TIGER-Lab/EvolveCoder-22K", split="train")
print(len(ds), ds[0]["id"], len(ds[0]["tests"]))


## Drop empty tests and duplicate statements

Also record the nine problems already generated that night, so the diagnostic slice can skip them. Printed: rows 21642, empty_tests 0, duplicate 0, kept 21642, peeked the nine ids below.


In [ ]:
import hashlib, json, random, re

def norm(s):
    return re.sub(r"\s+", " ", s).strip().lower()

seen, empty, dup, kept = set(), 0, 0, []
for r in ds:
    tests = [t for t in r["tests"] if str(t).strip()]
    if not tests:
        empty += 1
        continue
    key = hashlib.sha256(norm(r["question"]).encode()).hexdigest()
    if key in seen:
        dup += 1
        continue
    seen.add(key)
    kept.append(r["id"])
order = list(range(len(ds)))
random.Random(0).shuffle(order)
peeked = [ds[int(i)]["id"] for i in order[:8]] + ["primeintellect_4596"]
with open("/workspace/s3_corpus_ids.json", "w") as f:
    json.dump({"kept": kept, "peeked": peeked}, f)
    f.flush()
print("rows", len(ds))
print("empty_tests", empty)
print("duplicate", dup)
print("kept", len(kept))
print("peeked", peeked)


## Remove copies of LiveCodeBench v6

A row is a copy when its first 400 characters match an exam problem, or when it shares five 60-character stretches with one exam problem. The parquet is commit `a16d037`, not `main`. `datasets` rejects the old loader script.

Printed: exam 1055, dates 2023-05-07 to 2025-04-06, exam grams 1322616, leak_prefix 0, leak_ngram 946, kept 20696.


In [ ]:
import hashlib, json, os, re
from collections import defaultdict
os.environ["HF_HOME"] = "/workspace/hf-cache"
os.environ["HF_HUB_CACHE"] = "/workspace/hf-cache/hub"
from datasets import load_dataset
from huggingface_hub import hf_hub_download

_SPACE = re.compile(r"\s+")
def normalize(text):
    return _SPACE.sub(" ", text).strip().lower()
def prefix_hash(text):
    return hashlib.sha256(normalize(text)[:400].encode()).hexdigest()
def char_grams(text, n=60):
    body = normalize(text)
    if len(body) < n:
        return set()
    return {body[i:i + n] for i in range(len(body) - n + 1)}

rev = "a16d03780493b939b3601fb9da2ac3ed2b23caa2"
paths = [
    hf_hub_download(
        repo_id="livecodebench/code_generation_lite",
        filename=f"release_v6/test-{i:05d}-of-00009.parquet",
        repo_type="dataset",
        revision=rev,
    )
    for i in range(9)
]
lcb = load_dataset("parquet", data_files=paths, split="train")
lcb = lcb.remove_columns([c for c in lcb.column_names if c not in ("question_content", "question_id", "contest_date")])
print("exam", len(lcb), "dates", min(lcb["contest_date"]), max(lcb["contest_date"]), flush=True)

exam_q = list(lcb["question_content"])
prefixes = {prefix_hash(q) for q in exam_q}
gram_to_ids = defaultdict(set)
for i, q in enumerate(exam_q):
    for g in char_grams(q):
        gram_to_ids[g].add(i)
print("exam grams", len(gram_to_ids), flush=True)

leak_prefix = leak_ngram = 0
kept = []
for n, r in enumerate(ds):
    if prefix_hash(r["question"]) in prefixes:
        leak_prefix += 1
        continue
    hits = defaultdict(int)
    leaked = False
    for g in char_grams(r["question"]):
        for eid in gram_to_ids.get(g, ()):
            hits[eid] += 1
            if hits[eid] >= 5:
                leaked = True
                break
        if leaked:
            break
    if leaked:
        leak_ngram += 1
        continue
    kept.append(r["id"])
    if n and n % 2000 == 0:
        print("scanned", n, "kept", len(kept), flush=True)

prev = json.load(open("/workspace/s3_corpus_ids.json"))
with open("/workspace/s3_corpus_ids.json", "w") as f:
    json.dump({
        "kept": kept,
        "peeked": prev["peeked"],
        "empty_tests": 0,
        "duplicate": 0,
        "leak_prefix": leak_prefix,
        "leak_ngram": leak_ngram,
        "exam": "livecodebench release_v6 parquet",
        "exam_n": len(lcb),
    }, f)
    f.flush()
print("leak_prefix", leak_prefix)
print("leak_ngram", leak_ngram)
print("kept", len(kept))


## Freeze 40 diagnostic problems

Same kind of assert problem as training, never generated on. The nine peeked ids stay in the train pool. Printed: memory kept 20696, eval 40, train 20656, peeked still in train 9. Eval ids start `TACO_5353`, `TACO_9932`, `primeintellect_1347` and end `APPS_2555`.


In [ ]:
import json, random
print("memory kept", len(kept) if "kept" in globals() else None)
blob = json.load(open("/workspace/s3_corpus_ids.json"))
peeked = set(blob["peeked"])
pool = [i for i in kept if i not in peeked]
order = pool[:]
random.Random(1).shuffle(order)
eval_ids = order[:40]
eval_set = set(eval_ids)
train_ids = [i for i in kept if i not in eval_set]
out = {
    "kept": list(kept),
    "peeked": blob["peeked"],
    "eval": eval_ids,
    "train": train_ids,
    "empty_tests": 0,
    "duplicate": 0,
    "leak_prefix": 0,
    "leak_ngram": len(ds) - len(kept),
    "exam": "livecodebench release_v6 parquet",
    "exam_n": 1055,
}
with open("/workspace/s3_corpus_ids.json", "w") as f:
    json.dump(out, f)
    f.flush()
print("kept", len(kept))
print("eval", len(eval_ids))
print("train", len(train_ids))
print("peeked still in train", sum(i in set(train_ids) for i in peeked))
print("eval ids", eval_ids)


## Grade easiest-first until 200 keeps

Needs `/root/vllm-venv` and `reward.py` plus `sandbox.py` beside `grade_until.py`. The notebook kernel only builds the queue. vLLM grades four problems at a time, eight answers each.

Earlier grades already in `s3_solvability.json` are skipped: the short pile (14 keep / 25 drop of 39) and a random 200 (20 keep / 180 drop). This cell continued from about 34 keeps and printed `finished keeps 200 graded 558`.

Re-running after that file already holds 200 keeps does not grade again.


In [ ]:
import json, os, subprocess
from pathlib import Path

assert os.path.exists("/root/vllm-venv/bin/python")
script = Path("grade_until.py")
assert script.exists(), "run this notebook from the s3 directory"

def is_keep(rec):
    shaped = rec.get("shaped") or []
    return rec.get("status") == "keep" or any(x == 1.0 for x in shaped)

done = json.load(open("/workspace/s3_solvability.json"))
n_keep = sum(is_keep(v) for v in done.values())
if n_keep >= 200:
    print("already", n_keep, "keeps; not grading again")
else:
    blob = json.load(open("/workspace/s3_corpus_ids.json"))
    by_id = {r["id"]: r for r in ds}
    rows = []
    for i in blob["train"]:
        if i in done or i not in by_id:
            continue
        r = by_id[i]
        n_tests = len([t for t in r["tests"] if str(t).strip()])
        rows.append({
            "id": i,
            "question": r["question"],
            "tests": list(r["tests"]),
            "n_tests": n_tests,
            "chars": len(r["question"]),
        })
    rows.sort(key=lambda r: (r["n_tests"], r["chars"]))
    rows = rows[:4000]
    with open("/tmp/s3_until200.json", "w") as f:
        json.dump({"target": 200, "keeps_now": n_keep, "rows": rows}, f)
    print("keeps_now", n_keep, "queued", len(rows), "easiest_tests", rows[0]["n_tests"], "easiest_chars", rows[0]["chars"], flush=True)

    libs = subprocess.check_output(
        ["find", "/root/vllm-venv/lib/python3.12/site-packages/nvidia", "-type", "d", "-name", "lib"],
        text=True,
    ).split()
    env = os.environ.copy()
    env["LD_LIBRARY_PATH"] = ":".join(libs) + ":/usr/local/cuda/lib64"
    env["HF_HOME"] = "/workspace/hf-cache"
    env["HF_HUB_CACHE"] = "/workspace/hf-cache/hub"
    env["HF_HUB_OFFLINE"] = "1"
    env["VLLM_LOGGING_LEVEL"] = "WARNING"
    env["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    env["PYTHONUNBUFFERED"] = "1"
    subprocess.run(["/root/vllm-venv/bin/python", "-u", str(script.resolve())], check=True, env=env)


## Freeze the 200 rows

Printed `keeps 200 missing [] eval_leaks []`. None of the 40 diagnostic ids are in the file.


In [ ]:
import json

done = json.load(open("/workspace/s3_solvability.json"))
blob = json.load(open("/workspace/s3_corpus_ids.json"))
eval_ids = set(blob["eval"])

def is_keep(rec):
    shaped = rec.get("shaped") or []
    return rec.get("status") == "keep" or any(x == 1.0 for x in shaped)

keep_ids = sorted(i for i, rec in done.items() if is_keep(rec))
by_id = {r["id"]: r for r in ds}
missing = [i for i in keep_ids if i not in by_id]
rows = [{"id": i, "question": by_id[i]["question"], "tests": list(by_id[i]["tests"])} for i in keep_ids if i in by_id]
with open("/workspace/s3_train200.json", "w") as f:
    json.dump(rows, f)
    f.flush()
print("keeps", len(rows), "missing", missing, "eval_leaks", sorted(set(keep_ids) & eval_ids))


## Upload the private dataset

Printed `syedamaan/rufus-s3-coding-pilot 200`.


In [ ]:
import os
os.environ["HF_HOME"] = "/workspace/hf-cache"
os.environ["HF_HUB_CACHE"] = "/workspace/hf-cache/hub"
import json
from datasets import Dataset
from huggingface_hub import HfApi

repo = "syedamaan/rufus-s3-coding-pilot"
rows = json.load(open("/workspace/s3_train200.json"))
Dataset.from_list(rows).push_to_hub(repo, private=True)
card = """---
license: other
task_categories:
- text-generation
tags:
- code
size_categories:
- n<1K
---

# Rufus S3 coding pilot

200 EvolveCoder problems kept for a first GRPO pass on `syedamaan/qwen3-1.7b-rufus-s2-reasoning-rl`.

A problem is here because that checkpoint, at temperature 0.6, top_p 0.95, top_k 20, and 2048 new tokens, produced a mixed group of eight answers: at least one answer passed every hidden test, and the mean was at most 0.8. Problems solved every time, and problems never solved, were left out.

Selection was easy-first: fewest tests, then shortest question. This is not a random slice of EvolveCoder. The 40 diagnostic EvolveCoder ids are not included. LiveCodeBench is not included.

Columns: `id`, `question`, `tests`. The tests are the hidden asserts used as the reward.
"""
open("/tmp/s3_pilot_readme.md", "w").write(card)
HfApi().upload_file(
    path_or_fileobj="/tmp/s3_pilot_readme.md",
    path_in_repo="README.md",
    repo_id=repo,
    repo_type="dataset",
)
print(repo, len(rows))
